<a href="https://colab.research.google.com/github/shetty-23/Computer-vision---Assignment-1/blob/feature%2Ftask-2-deblurring/Assignment_1_Computer_vision.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
import os
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split


In [24]:
# 1. Setup Paths
base_path = Path("/content/drive/MyDrive/gopro_deblur")
# Pointing to the actual subdirectories containing the .png files
blur_dir = base_path / "blur" / "images"
sharp_dir = base_path / "sharp" / "images"

In [25]:
# Searching for .png files in the nested 'images' directory
filenames = sorted([f.name for f in blur_dir.glob("*.png")])

if len(filenames) == 0:
    print(f"No .png files found in {blur_dir}. Please check your path.")
else:
    print(f"Found {len(filenames)} files. Splitting into train, val, and test...")
    train_files, temp_files = train_test_split(filenames, test_size=0.3, random_state=42)
    val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)
    print(f"Splits created: Train({len(train_files)}), Val({len(val_files)}), Test({len(test_files)})")

def move_to_split(file_list, split_name):
    for fname in file_list:
        for folder in ["blur", "sharp"]:
            dest = base_path / "processed" / split_name / folder
            dest.mkdir(parents=True, exist_ok=True)

            # Source files are in the 'images' subfolder
            src_file = base_path / folder / "images" / fname
            if src_file.exists():
                shutil.copy(src_file, dest / fname)
            else:
                print(f"Warning: Source file not found: {src_file}")

Found 1029 files. Splitting into train, val, and test...
Splits created: Train(720), Val(154), Test(155)


In [29]:
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

def get_blur_score(image_path):
    image = cv2.imread(str(image_path))
    if image is None: return 0
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

print("Calculating blur scores...")
scores = []
for fname in tqdm(filenames):
    path = blur_dir / fname
    scores.append(get_blur_score(path))

# Create DataFrame for analysis
df_blur = pd.DataFrame({'filename': filenames, 'score': scores})

# Define thresholds for categories (Quantile-based for even distribution)
# You can adjust these fixed values if you prefer specific variance ranges
low, high = df_blur['score'].quantile([0.33, 0.66])

def categorize(s):
    if s <= low: return 'Severe'
    if s <= high: return 'Medium'
    return 'Mild'

df_blur['category'] = df_blur['score'].apply(categorize)
print("\nBlur Category Distribution:")
print(df_blur['category'].value_counts())

# Perform Stratified Split (70/15/15)
train_list, val_list, test_list = [], [], []

for cat in ['Mild', 'Medium', 'Severe']:
    cat_files = df_blur[df_blur['category'] == cat]['filename'].tolist()
    tr, temp = train_test_split(cat_files, test_size=0.3, random_state=42)
    vl, ts = train_test_split(temp, test_size=0.5, random_state=42)
    train_list.extend(tr)
    val_list.extend(vl)
    test_list.extend(ts)

print(f"\nFinal Stratified Splits: Train({len(train_list)}), Val({len(val_list)}), Test({len(test_list)})")

Calculating blur scores...


100%|██████████| 1029/1029 [01:06<00:00, 15.47it/s]



Blur Category Distribution:
category
Mild      350
Severe    340
Medium    339
Name: count, dtype: int64

Final Stratified Splits: Train(720), Val(154), Test(155)


In [ ]:
def move_stratified_files(df, split_list, split_name):
    print(f"Moving {split_name} files...")
    for fname in tqdm(split_list):
        # Get category for this specific file
        category = df[df['filename'] == fname]['category'].values[0]

        for folder_type in ["blur", "sharp"]:
            # New path structure: processed / split / category / folder_type
            dest = base_path / "processed_stratified" / split_name / category / folder_type
            dest.mkdir(parents=True, exist_ok=True)

            src_file = base_path / folder_type / "images" / fname
            if src_file.exists():
                shutil.copy(src_file, dest / fname)

# Execute the move for each split
move_stratified_files(df_blur, train_list, 'train')
move_stratified_files(df_blur, val_list, 'val')
move_stratified_files(df_blur, test_list, 'test')

print("\nAll files have been organized into stratified folders.")

Moving train files...


 34%|███▍      | 244/720 [01:17<02:12,  3.60it/s]